# xView Dataset EDA
Exploratory analysis before preprocessing.
Run on EC2 with xView data pulled from S3.

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

# Paths — match ec2_eda.sh layout
DATA_DIR    = Path('/home/ec2-user/argus/data/raw')
LABELS_PATH = DATA_DIR / 'train_labels/xView_train.geojson'
IMAGES_DIR  = DATA_DIR / 'train_images/train_images'
PLOTS_DIR   = Path('/home/ec2-user/argus/notebooks/plots')
PLOTS_DIR.mkdir(exist_ok=True)

print('Labels:', LABELS_PATH.exists())
print('Images dir:', IMAGES_DIR.exists())

## 1. Load GeoJSON

In [ ]:
with open(LABELS_PATH) as f:
    geojson = json.load(f)

features = geojson['features']
print(f'Total annotated objects: {len(features):,}')
print(f'Sample feature keys: {list(features[0]["properties"].keys())}')

## 2. xView class distribution (all 60 classes)

In [ ]:
type_ids = [f['properties'].get('type_id', -1) for f in features]
class_counts = Counter(type_ids)

ids   = sorted(class_counts.keys())
counts = [class_counts[i] for i in ids]

fig, ax = plt.subplots(figsize=(18, 4))
ax.bar([str(i) for i in ids], counts)
ax.set_xlabel('xView type_id')
ax.set_ylabel('Object count')
ax.set_title('Object count per xView class')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'xview_class_dist.png', dpi=120)
plt.show()
print(f'Unique classes: {len(class_counts)}')

## 3. ARGUS 5-class distribution

In [ ]:
import sys
sys.path.insert(0, '/home/ec2-user/argus')
from src.preprocessing.xview_converter import XVIEW_TO_ARGUS, CLASS_NAMES

argus_counts = Counter()
dropped = 0
for f in features:
    tid = f['properties'].get('type_id', -1)
    if tid in XVIEW_TO_ARGUS:
        argus_counts[XVIEW_TO_ARGUS[tid]] += 1
    else:
        dropped += 1

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, [argus_counts[i] for i in range(5)], color='steelblue')
ax.bar_label(bars)
ax.set_ylabel('Object count')
ax.set_title('ARGUS 5-class distribution')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'argus_class_dist.png', dpi=120)
plt.show()

total_kept = sum(argus_counts.values())
print(f'Kept: {total_kept:,}  |  Dropped (unmapped): {dropped:,}  |  Keep rate: {total_kept/(total_kept+dropped):.1%}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {argus_counts[i]:,}')

## 4. Bounding box size distribution

In [ ]:
box_widths  = []
box_heights = []

for f in features:
    if f['properties'].get('type_id', -1) not in XVIEW_TO_ARGUS:
        continue
    coords = f['geometry']['coordinates'][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    box_widths.append(max(xs) - min(xs))
    box_heights.append(max(ys) - min(ys))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(box_widths,  bins=80, color='steelblue')
axes[0].set_xlabel('Box width (pixels)')
axes[0].set_title('Bounding box widths')
axes[1].hist(box_heights, bins=80, color='salmon')
axes[1].set_xlabel('Box height (pixels)')
axes[1].set_title('Bounding box heights')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'bbox_sizes.png', dpi=120)
plt.show()

print(f'Width  — mean: {np.mean(box_widths):.1f}  median: {np.median(box_widths):.1f}  max: {np.max(box_widths):.1f}')
print(f'Height — mean: {np.mean(box_heights):.1f}  median: {np.median(box_heights):.1f}  max: {np.max(box_heights):.1f}')

## 5. Images per class — which images have tanks vs aircraft etc.

In [ ]:
# Count distinct images that contain at least one object of each ARGUS class
images_with_class = {i: set() for i in range(5)}

for f in features:
    tid = f['properties'].get('type_id', -1)
    if tid not in XVIEW_TO_ARGUS:
        continue
    img_id = f['properties'].get('image_id', '')
    images_with_class[XVIEW_TO_ARGUS[tid]].add(img_id)

print('Images containing each class:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {len(images_with_class[i])} images')

## 6. Sample images with bounding boxes

In [ ]:
CLASS_COLORS = ['red', 'orange', 'blue', 'green', 'purple']

by_image = {}
for f in features:
    img_id = f['properties'].get('image_id', '')
    by_image.setdefault(img_id, []).append(f)

valid_ids = [
    img_id for img_id, feats in by_image.items()
    if (IMAGES_DIR / img_id).exists()
    and any(f['properties'].get('type_id', -1) in XVIEW_TO_ARGUS for f in feats)
]
sample_ids = random.sample(valid_ids, min(6, len(valid_ids)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_id in zip(axes, sample_ids):
    img = Image.open(IMAGES_DIR / img_id)
    ax.imshow(img)
    ax.set_title(img_id, fontsize=8)
    ax.axis('off')

    for f in by_image[img_id]:
        tid = f['properties'].get('type_id', -1)
        if tid not in XVIEW_TO_ARGUS:
            continue
        cls = XVIEW_TO_ARGUS[tid]
        coords = f['geometry']['coordinates'][0]
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        x0, y0 = min(xs), min(ys)
        w,  h  = max(xs) - x0, max(ys) - y0
        rect = patches.Rectangle((x0, y0), w, h,
                                  linewidth=1, edgecolor=CLASS_COLORS[cls], facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0 - 2, CLASS_NAMES[cls], color=CLASS_COLORS[cls], fontsize=6)

from matplotlib.lines import Line2D
legend = [Line2D([0],[0], color=c, lw=2, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=legend, loc='lower center', ncol=5, fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sample_images.png', dpi=120)
plt.show()

## 7. Class imbalance ratio — key for training

In [ ]:
counts = [argus_counts[i] for i in range(5)]
max_count = max(counts)

print('Class imbalance (ratio to most common class):')
for name, cnt in zip(CLASS_NAMES, counts):
    ratio = max_count / cnt if cnt > 0 else float('inf')
    bar = '█' * int(cnt / max_count * 40)
    print(f'  {name:15s}: {cnt:6,}  (1:{ratio:.1f})  {bar}')

print()
print('Classes with ratio > 10x are severely underrepresented.')
print('Consider: class weights in loss, oversampling, or more data sources.')

## 8. Image size distribution

In [ ]:
all_img_files = list(IMAGES_DIR.glob('*.tif'))[:50]
widths, heights = [], []

for p in all_img_files:
    try:
        img = Image.open(p)
        widths.append(img.width)
        heights.append(img.height)
    except Exception:
        pass

print(f'Sampled {len(widths)} images')
print(f'Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}')

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(widths,  bins=30, color='steelblue')
axes[0].set_title('Image widths')
axes[1].hist(heights, bins=30, color='salmon')
axes[1].set_title('Image heights')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'image_sizes.png', dpi=120)
plt.show()

## Summary

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Total objects in xView:    {len(features):,}')
print(f'Objects mapped to ARGUS:   {total_kept:,}  ({total_kept/(total_kept+dropped):.1%})')
print(f'Objects dropped:           {dropped:,}')
print()
print('Per-class counts:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {argus_counts[i]:,}')
print()
print('Median box size:', f'{np.median(box_widths):.1f} x {np.median(box_heights):.1f} px')
print('Plots saved: xview_class_dist.png, argus_class_dist.png, bbox_sizes.png, sample_images.png, image_sizes.png')